In [4]:
import os
import numpy as np
import keras
from keras import layers, models, ops

# ResNet50 Building Blocks
def identity_block(x, filters, kernel_size=3, name=None):
    """Identity block for ResNet"""
    f1, f2, f3 = filters

    shortcut = x

    # First layer
    x = layers.Conv2D(f1, 1, padding='same', name=f'{name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Activation('relu', name=f'{name}_relu1')(x)

    # Second layer
    x = layers.Conv2D(f2, kernel_size, padding='same', name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    x = layers.Activation('relu', name=f'{name}_relu2')(x)

    # Third layer
    x = layers.Conv2D(f3, 1, padding='same', name=f'{name}_conv3')(x)
    x = layers.BatchNormalization(name=f'{name}_bn3')(x)

    # Add shortcut
    x = layers.Add(name=f'{name}_add')([x, shortcut])
    x = layers.Activation('relu', name=f'{name}_relu3')(x)

    return x

def conv_block(x, filters, kernel_size=3, stride=2, name=None):
    """Convolutional block for ResNet"""
    f1, f2, f3 = filters

    shortcut = x

    # First layer
    x = layers.Conv2D(f1, 1, strides=stride, padding='same', name=f'{name}_conv1')(x)
    x = layers.BatchNormalization(name=f'{name}_bn1')(x)
    x = layers.Activation('relu', name=f'{name}_relu1')(x)

    # Second layer
    x = layers.Conv2D(f2, kernel_size, padding='same', name=f'{name}_conv2')(x)
    x = layers.BatchNormalization(name=f'{name}_bn2')(x)
    x = layers.Activation('relu', name=f'{name}_relu2')(x)

    # Third layer
    x = layers.Conv2D(f3, 1, padding='same', name=f'{name}_conv3')(x)
    x = layers.BatchNormalization(name=f'{name}_bn3')(x)

    # Shortcut projection
    shortcut = layers.Conv2D(f3, 1, strides=stride, padding='same', name=f'{name}_shortcut_conv')(shortcut)
    shortcut = layers.BatchNormalization(name=f'{name}_shortcut_bn')(shortcut)

    # Add shortcut
    x = layers.Add(name=f'{name}_add')([x, shortcut])
    x = layers.Activation('relu', name=f'{name}_relu3')(x)

    return x

def build_resnet50_small(input_shape=(128, 128, 3), num_classes=10):
    """
    Build a lighter ResNet50 optimized for SMALL DATASETS
    Reduces parameters by ~70% while keeping ResNet architecture
    """
    inputs = layers.Input(shape=input_shape)

    # Initial convolution - smaller filters
    x = layers.Conv2D(32, 5, strides=2, padding='same', name='conv1')(inputs)
    x = layers.BatchNormalization(name='bn_conv1')(x)
    x = layers.Activation('relu', name='relu_conv1')(x)
    x = layers.MaxPooling2D(3, strides=2, padding='same', name='pool1')(x)

    # Stage 2 - Reduced from [64,64,256] to [32,32,128]
    x = conv_block(x, [32, 32, 128], stride=1, name='stage2_block1')
    x = identity_block(x, [32, 32, 128], name='stage2_block2')

    # Stage 3 - Reduced from [128,128,512] to [64,64,256]
    x = conv_block(x, [64, 64, 256], stride=2, name='stage3_block1')
    x = identity_block(x, [64, 64, 256], name='stage3_block2')
    x = identity_block(x, [64, 64, 256], name='stage3_block3')

    # Stage 4 - Reduced from [256,256,1024] to [128,128,512]
    x = conv_block(x, [128, 128, 512], stride=2, name='stage4_block1')
    x = identity_block(x, [128, 128, 512], name='stage4_block2')
    x = identity_block(x, [128, 128, 512], name='stage4_block3')

    # Stage 5 - Reduced from [512,512,2048] to [256,256,1024]
    x = conv_block(x, [256, 256, 1024], stride=2, name='stage5_block1')
    x = identity_block(x, [256, 256, 1024], name='stage5_block2')

    # Classification head with more regularization
    x = layers.GlobalAveragePooling2D(name='avg_pool')(x)
    x = layers.Dropout(0.6)(x)
    x = layers.Dense(256, activation='relu', name='fc1')(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

    model = models.Model(inputs=inputs, outputs=outputs, name='resnet50_small')
    return model

# Data augmentation layer - AGGRESSIVE for small datasets
def get_augmentation_model():
    """Create strong data augmentation for small datasets"""
    augmentation = keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.2),  # Increased

    ], name='augmentation')
    return augmentation

# Dataset path
dataset_path = '/content/drive/MyDrive/concave'

# Get number of classes
class_names = sorted([d for d in os.listdir(dataset_path)
                      if os.path.isdir(os.path.join(dataset_path, d))])
num_classes = len(class_names)
print(f"Found {num_classes} classes: {class_names}")

# Count total images
total_images = sum([len(os.listdir(os.path.join(dataset_path, c)))
                    for c in class_names])
print(f"Total images: {total_images}")

# Use smaller image size for small datasets (faster + less overfitting)
IMG_SIZE = 128  # Reduced from 224
BATCH_SIZE = 16  # Smaller batch for small datasets

# Load training data
train_dataset = keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='training',
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Load validation data
val_dataset = keras.utils.image_dataset_from_directory(
    dataset_path,
    validation_split=0.2,
    subset='validation',
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

# Normalize images
normalization_layer = layers.Rescaling(1./255)
train_dataset = train_dataset.map(lambda x, y: (normalization_layer(x), y))
val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

# Cache and prefetch for performance
train_dataset = train_dataset.cache().prefetch(buffer_size=32)
val_dataset = val_dataset.cache().prefetch(buffer_size=32)

# Build model with augmentation
augmentation = get_augmentation_model()
inputs = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = augmentation(inputs)
base_model = build_resnet50_small(input_shape=(IMG_SIZE, IMG_SIZE, 3), num_classes=num_classes)
outputs = base_model(x)

model = models.Model(inputs=inputs, outputs=outputs)

# Compile with L2 regularization via weight decay in optimizer
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=0.0001,
        weight_decay=0.0001  # L2 regularization
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_acc')]
)

# Print model summary
print("\nModel Summary:")
total_params = sum([np.prod(v.shape) for v in model.trainable_weights])
print(f"Total trainable parameters: {total_params:,}")
model.summary()

# Callbacks optimized for small datasets
callbacks = [
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=15,  # More patience for small datasets
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=7,
        min_lr=1e-7,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        'best_resnet50_small.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    )
]

# Train model
print("\n" + "="*50)
print("Starting training (optimized for SMALL datasets)")
print("="*50)
history = model.fit(
    train_dataset,
    epochs=100,  # More epochs with early stopping
    validation_data=val_dataset,
    callbacks=callbacks,
    verbose=1
)

# Save final model
model.save('final_resnet50_small.keras')

print("\n" + "="*50)
print("Training completed!")
print("="*50)
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.4f}")
print(f"Model saved as: best_resnet50_small.keras")

Found 6 classes: ['Moon_jellyfish', 'barrel_jellyfish', 'blue_jellyfish', 'compass_jellyfish', 'lions_mane_jellyfish', 'mauve_stinger_jellyfish']
Total images: 900
Found 900 files belonging to 6 classes.
Using 720 files for training.
Found 900 files belonging to 6 classes.
Using 180 files for validation.

Model Summary:
Total trainable parameters: 4,117,574


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ augmentation (Sequential)       │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resnet50_small (Functional)     │ (None, 6)              │     4,135,302 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,135,302 (15.77 MB)

 Trainable params: 4,117,574 (15.71 MB)

 Non-trainable params: 17,728 (69.25 KB)


Starting training (optimized for SMALL datasets)
Epoch 1/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.1986 - loss: 3.2066 - top3_acc: 0.5303
Epoch 1: val_accuracy improved from -inf to 0.20000, saving model to best_resnet50_small.keras
45/45 ━━━━━━━━━━━━━━━━━━━━ 183s 4s/step - accuracy: 0.1985 - loss: 3.2011 - top3_acc: 0.5306 - val_accuracy: 0.2000 - val_loss: 1.7834 - val_top3_acc: 0.5222 - learning_rate: 1.0000e-04
Epoch 2/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step - accuracy: 0.1932 - loss: 2.6452 - top3_acc: 0.5317
Epoch 2: val_accuracy did not improve from 0.20000
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 66ms/step - accuracy: 0.1931 - loss: 2.6443 - top3_acc: 0.5318 - val_accuracy: 0.2000 - val_loss: 1.7829 - val_top3_acc: 0.5444 - learning_rate: 1.0000e-04
Epoch 3/100
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step - accuracy: 0.2390 - loss: 2.2128 - top3_acc: 0.5894
Epoch 3: val_accuracy did not improve from 0.20000
45/45 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - accuracy: 0.2389 - loss: 2